# Forge Vector Database Setup

Store generated embeddings and metadata inside a vector database to enable semantic search and retrieval.

This notebook creates the searchable knowledge layer required for the RAG pipeline.

In [1]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 6.5 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import time

import numpy as np

from google.colab import drive

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)


In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_ROOT = Path("/content/drive/MyDrive/forge")

CONFIG = {
    "project_root": PROJECT_ROOT,
    "embeddings": PROJECT_ROOT / "knowledge_base" / "embeddings.npy",
    "metadata": PROJECT_ROOT / "knowledge_base" / "embedding_metadata.json",
    "vector_db": PROJECT_ROOT / "knowledge_base" / "vector_store"
}

print("Project Root:", CONFIG["project_root"])
print("Embeddings:", CONFIG["embeddings"])
print("Metadata:", CONFIG["metadata"])

Project Root: /content/drive/MyDrive/forge
Embeddings: /content/drive/MyDrive/forge/knowledge_base/embeddings.npy
Metadata: /content/drive/MyDrive/forge/knowledge_base/embedding_metadata.json


In [5]:
embeddings = np.load(CONFIG["embeddings"])

with open(CONFIG["metadata"], "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Embeddings shape:", embeddings.shape)
print("Metadata count:", len(metadata))

Embeddings shape: (3944, 768)
Metadata count: 3944


In [6]:
qdrant_path = CONFIG["vector_db"]

qdrant_path.mkdir(parents=True, exist_ok=True)

client = QdrantClient(
    path=str(qdrant_path)
)

print("Qdrant initialized successfully.")
print("Storage:", qdrant_path)

Qdrant initialized successfully.
Storage: /content/drive/MyDrive/forge/knowledge_base/vector_store


In [7]:
COLLECTION_NAME = "forge_knowledge"

vector_size = embeddings.shape[1]

# Remove old collection if it exists (for clean setup)
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE
    )
)

print("Collection created:", COLLECTION_NAME)
print("Vector size:", vector_size)

Collection created: forge_knowledge
Vector size: 768


In [8]:
points = []

for idx, (vector, meta) in enumerate(zip(embeddings, metadata)):
    points.append(
        PointStruct(
            id=idx,
            vector=vector.tolist(),
            payload=meta
        )
    )

print(f"Prepared {len(points)} points")

Prepared 3944 points


In [9]:
start_time = time.time()

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

end_time = time.time()

print("Vectors uploaded successfully.")
print(f"Inserted points: {len(points)}")
print(f"Time taken: {(end_time - start_time):.2f} seconds")

Vectors uploaded successfully.
Inserted points: 3944
Time taken: 136.40 seconds


In [10]:
collection_info = client.get_collection(
    collection_name=COLLECTION_NAME
)

print("Collection:", COLLECTION_NAME)
print("Vectors stored:", collection_info.points_count)
print("Vector size:", collection_info.config.params.vectors.size)

Collection: forge_knowledge
Vectors stored: 3944
Vector size: 768


In [12]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-base-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [15]:
query = "Which vector database should I use for a RAG application?"

# Create query embedding
query_embedding = model.encode(
    query,
    convert_to_numpy=True
)

# Search Qdrant
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding.tolist(),
    limit=5
)

print(f"Results found: {len(results.points)}")
print()

for i, result in enumerate(results.points):
    print("=" * 60)
    print(f"Result {i+1}")
    print("Score:", result.score)
    print("Technology:", result.payload["technology"])
    print("Source:", result.payload["source"])
    print("Text preview:")
    print(result.payload["text"][:300])
    print()

Results found: 5

Result 1
Score: 0.7093405045999259
Technology: ragas
Source: research_papers
Text preview:
. With Ragas, we put forward a suite of metrics which can be used to evaluate these different dimensions \textit{without having to rely on ground truth human annotations}. We posit that such a framework can crucially contribute to faster evaluation cycles of RAG architectures, which is especially im

Result 2
Score: 0.6989406644668712
Technology: weaviate
Source: github_repository
Text preview:
-
🔌 Flexible Vectorization: Seamlessly vectorize data at import time with integrated vectorizers from OpenAI, Cohere, HuggingFace, Google, and more. Or you can import your own vector embeddings.
-
🔍 Advanced Hybrid & Image Search: Combine the power of semantic search with traditional keyword (BM25) 

Result 3
Score: 0.6966195239515824
Technology: milvus
Source: github_repository
Text preview:
🐦 Milvus is a high-performance vector database built for scale. It powers AI applications by effi